# 08 — Final models on the test set

Each member's final model, fitted on train and scored once on the shared test set by AUC.

> **TODO (Oscar):** a line or two on what this notebook is for and the rule that nothing is tuned here.

In [1]:
import importlib.util
import sys
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
import numpy as np
import pandas as pd
from scipy.stats import bootstrap
import statsmodels.api as sm
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

ROOT = Path.cwd().parent                  # run from report/
PROCESSED = ROOT / "data" / "processed"
TARGET = "default payment next month"

train = pd.read_csv(PROCESSED / "taiwan-train.csv", index_col="ID")
test  = pd.read_csv(PROCESSED / "taiwan-test.csv", index_col="ID")
y_train, y_test = train[TARGET], test[TARGET]

results = {}                              # model name -> test AUC

def test_auc(name, p_test):
    """Score one model's test-set probabilities of default."""
    results[name] = roc_auc_score(y_test, p_test)
    print(f"{name:<30} test AUC {results[name]:.4f}")

print(f"train {len(train):,}   test {len(test):,} customers, {y_test.mean():.2%} defaulted")

train 17,997   test 5,999 customers, 22.12% defaulted


## 1. Logistic regression

In [2]:
sys.path.append(str(ROOT / "Oscar"))
import prep                               # Oscar/prep.py: fixes 1-4 + ratio features

X_train = prep.prepare(train.drop(columns=TARGET), ratios=True)
X_test  = prep.prepare(test.drop(columns=TARGET), ratios=True)

# final model from 05: L2, C = 0.0316 (tuned on CV log loss)
logistic = make_pipeline(StandardScaler(),
                         LogisticRegression(C=0.0316, l1_ratio=0, solver="liblinear", max_iter=1000))
logistic.fit(X_train, y_train);

In [3]:
test_auc("Logistic regression", logistic.predict_proba(X_test)[:, 1])

Logistic regression            test AUC 0.7872


## 2. SVM

In [4]:
# final model from 06: linear SVM, squared hinge, C = 1e-4, Platt scaling for probabilities
# same features as the logistic regression above
svm = make_pipeline(StandardScaler(),
                    CalibratedClassifierCV(LinearSVC(C=1e-4, dual=True, max_iter=20000, random_state=2026),
                                           method="sigmoid", cv=5))
svm.fit(X_train, y_train);

In [5]:
test_auc("SVM", svm.predict_proba(X_test)[:, 1])

SVM                            test AUC 0.7903


## 3. Logit hazard

In [6]:
# Alex/modelling/prep.py builds the person-month table; loaded under another name, as Oscar's prep is already imported
spec = importlib.util.spec_from_file_location("hazard_prep", ROOT / "Alex" / "modelling" / "prep.py")
hazard_prep = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hazard_prep)

pm = hazard_prep.person_months({"train": train, "test": test})
FEATURES = hazard_prep.features(pm["train"])


def fit_hazard(rows):
    return sm.GLM(rows["event"], sm.add_constant(rows[FEATURES]), family=sm.families.Binomial()).fit()


def october_hazard(model, rows):
    """The hazard in October, i.e. the probability of default, for each customer."""
    october = hazard_prep.october(rows)
    return model.predict(sm.add_constant(october[FEATURES], has_constant="add"))


# final model from 07: logit hazard, with Platt scaling fitted on out-of-fold training predictions
hazard = fit_hazard(pm["train"])
folds = GroupKFold(n_splits=10, shuffle=True, random_state=2026).split(pm["train"], groups=pm["train"].index)
out_of_fold = pd.concat([october_hazard(fit_hazard(pm["train"].iloc[fit_idx]), pm["train"].iloc[score_idx])
                         for fit_idx, score_idx in folds])
log_odds = lambda p: np.log(p / (1 - p)).to_frame("log-odds")
platt = LogisticRegression().fit(log_odds(out_of_fold), y_train.loc[out_of_fold.index])

In [7]:
test_auc("Logit hazard", platt.predict_proba(log_odds(october_hazard(hazard, pm["test"])))[:, 1])

Logit hazard                   test AUC 0.7504


## 4. Markov Chain

In [8]:
validation = pd.read_csv(ROOT / "data" / "processed" / "taiwan-validation.csv", index_col="ID")
pay_cols  = [f"PAY_{i}" for i in range(6, 0, -1)]          # chronological Apr..Sep
bill_cols = [f"BILL_AMT{i}" for i in range(6, 0, -1)]
amt_cols  = [f"PAY_AMT{i}" for i in range(6, 0, -1)]
y_tr, y_va, y_te = train[TARGET].values, validation[TARGET].values, test[TARGET].values
STATE_SPECS = {
    "mk3": dict(
        map={-2: 0, -1: 0, 0: 1},                        # current/paid, revolving
        top=1,                                            # every status >= 1 month late -> one state
        labels=["current/paid", "revolving", "late (1+)"],
    ),
}

def _to_state_fn(mapping, top):
    last = max(mapping.values()) + 1
    raw_vals = list(mapping)
    def f(p):
        p = np.asarray(p)
        s = np.full(p.shape, last, dtype=int)
        for raw, st in mapping.items():
            s[p == raw] = st
        bad = (p < top) & ~np.isin(p, raw_vals)
        if bad.any():
            raise ValueError(f"unmapped PAY values: {np.unique(p[bad])}")
        return s
    return f

def make_all_states(specs=STATE_SPECS):
    out = {}
    for name, sp in specs.items():
        K = max(sp["map"].values()) + 2          # mapped states + one final "top" state
        labels = sp.get("labels")
        if labels is not None and len(labels) != K:
            raise ValueError(f"{name}: {len(labels)} labels but K={K}")
        out[name] = dict(to_state=_to_state_fn(sp["map"], sp["top"]), K=K, labels=labels)
    return out

states = make_all_states(STATE_SPECS)
base = states["mk3"]

def debt_matrix(d, measure="util"):
    bill = np.clip(d[bill_cols].values.astype(float), 0, None)       # chronological Apr..Sep
    return bill / d["LIMIT_BAL"].values[:, None] if measure == "util" else bill

def fit_chain_aug(S, K, alpha=1.0):
    init = np.bincount(S[:, 0], minlength=K) + alpha
    C = np.full((K, K), alpha)
    np.add.at(C, (S[:, :-1].ravel(), S[:, 1:].ravel()), 1)
    return init / init.sum(), C / C.sum(1, keepdims=True)

def llr_aug(S, m):
    sc = np.log(m["init1"][S[:, 0]]) - np.log(m["init0"][S[:, 0]])
    for t in range(S.shape[1] - 1):
        sc += np.log(m["P1"][S[:, t], S[:, t+1]]) - np.log(m["P0"][S[:, t], S[:, t+1]])
    return sc + m["log_prior"]

def build_markov_debt(name, base_to_state, K_base, base_labels=None, n_bins=3,
                      measure="util", split=None, alpha=1.0):
    # debt-bin edges from TRAIN only
    D_tr = debt_matrix(train, measure)
    edges = np.unique(np.quantile(D_tr.ravel(), np.linspace(0, 1, n_bins + 1)[1:-1]))
    nb = len(edges) + 1
    split = set(range(K_base) if split is None else split)   # base states to subdivide by debt

    lookup, labels, k = np.zeros((K_base, nb), dtype=int), [], 0
    for s in range(K_base):
        base = base_labels[s] if base_labels else str(s)
        if s in split:
            for b in range(nb):
                lookup[s, b] = k; k += 1; labels.append(f"{base} | debt {b+1}/{nb}")
        else:
            lookup[s, :] = k; k += 1; labels.append(base)
    K = k

    def encode(d):
        s = base_to_state(d[pay_cols].values)
        return lookup[s, np.digitize(debt_matrix(d, measure), edges)]

    S_tr, S_va, S_te = encode(train), encode(validation), encode(test)
    ytr = np.asarray(y_tr).astype(int)
    init0, P0 = fit_chain_aug(S_tr[ytr == 0], K, alpha)       # TRAIN only
    init1, P1 = fit_chain_aug(S_tr[ytr == 1], K, alpha)
    m = dict(name=name, K=K, labels=labels, edges=edges, init0=init0, P0=P0, init1=init1, P1=P1,
             log_prior=np.log(ytr.mean() / (1 - ytr.mean())))
    m["S_tr"], m["y_tr"] = S_tr, y_tr
    m["sc_tr"], m["sc_va"], m["sc_te"] = llr_aug(S_tr, m), llr_aug(S_va, m), llr_aug(S_te, m)
    m["platt"] = LogisticRegression(C=1e6).fit(m["sc_va"].reshape(-1, 1), y_va)   # VALIDATION only
    return m

# debt model: 3 payment states x 3 debt bins = 9 states
mk3_debt = build_markov_debt("mk3_debt", base["to_state"], base["K"],
                             base_labels=base["labels"], n_bins=3, measure="util")

test_auc("Markov debt", mk3_debt["platt"].predict_proba(mk3_debt["sc_te"].reshape(-1, 1))[:, 1])

Markov debt                    test AUC 0.7655


## 5. Random Forest and  HGB

In [9]:
SEED = 42
def make_features(d):
    X = d.drop(columns=[TARGET]).copy()
    L = X["LIMIT_BAL"]
    B, A = X[bill_cols].values, X[amt_cols].values
    X["n_late"]      = (X[pay_cols] > 0).sum(axis=1)
    X["max_delay"]   = X[pay_cols].max(axis=1)
    X["mean_pay"]    = X[pay_cols].mean(axis=1)
    X["recent_late"] = (X[["PAY_1", "PAY_2"]] > 0).sum(axis=1)
    X["n_revolve"]   = (X[pay_cols] == 0).sum(axis=1)
    X["pay_slope"]   = X["PAY_1"] - X["PAY_6"]
    X["util_1"]      = B[:, -1] / L
    X["util_mean"]   = B.mean(axis=1) / L
    X["util_max"]    = B.max(axis=1) / L
    X["bill_trend"]  = (B[:, -1] - B[:, 0]) / L
    X["repay_ratio"] = A.sum(axis=1) / (np.clip(B, 0, None).sum(axis=1) + 1)
    X["zero_pay_n"]  = (A == 0).sum(axis=1)
    X["bill_vol"]    = B.std(axis=1) / (np.abs(B).mean(axis=1) + 1)
    return X

X_tr, X_va, X_te = make_features(train), make_features(validation), make_features(test)

# Final RF: best params from RandomizedSearchCV (CV AUC 0.7805), refit with 500 trees
rf_best = RandomForestClassifier(
    min_samples_leaf=59, max_features=0.35, max_depth=12, max_samples=0.7,
    n_estimators=500, n_jobs=-1, random_state=SEED,
).fit(X_tr, y_tr)

# Final HGB: best params from RandomizedSearchCV (CV AUC 0.7816); early stopping picks the iteration count
hgb_best = HistGradientBoostingClassifier(
    l2_regularization=1.7912362571043667, learning_rate=0.04857876818821363,
    max_depth=4, max_leaf_nodes=28, min_samples_leaf=149,
    max_iter=500, early_stopping=True, n_iter_no_change=15, random_state=SEED,
).fit(X_tr, y_tr)

models = {"RF": rf_best, "HGB": hgb_best}
for n, m in models.items():
    test_auc(f"{n} (final)", m.predict_proba(X_te)[:, 1])

RF (final)                     test AUC 0.7968
HGB (final)                    test AUC 0.7970


## Results

In [10]:
pd.Series(results, name="test AUC").sort_values(ascending=False).round(4)

HGB (final)            0.7970
RF (final)             0.7968
SVM                    0.7903
Logistic regression    0.7872
Markov debt            0.7655
Logit hazard           0.7504
Name: test AUC, dtype: float64

## 6. Within-cluster testing

A single AUC can hide how well a model ranks customers of the same type (as seen in `07-HazardAndClusteringModels.ipynb`,
section 8). So, we put the test customers into the same five clusters as in 07, built from the training customers and score each model within each cluster. The ± is half the width of a 95% bootstrap
interval over customers.

In [11]:
TYPES = {0: "low use, pays off", 1: "carries a balance, pays little", 2: "inactive",
         3: "often behind", 4: "growing balance"}             # as described in 07
feats = {"train": hazard_prep.customer_features(train), "test": hazard_prep.customer_features(test)}
scaler = StandardScaler().fit(feats["train"])
km = KMeans(n_clusters=5, n_init=20, random_state=2026).fit(scaler.transform(feats["train"]))
cluster_test = pd.Series(km.predict(scaler.transform(feats["test"])), index=test.index)

p_test = {"Logistic regression": logistic.predict_proba(X_test)[:, 1],
          "SVM": svm.predict_proba(X_test)[:, 1],
          "Logit hazard": platt.predict_proba(log_odds(october_hazard(hazard, pm["test"]).loc[test.index]))[:, 1],
          "Markov debt": mk3_debt["platt"].predict_proba(mk3_debt["sc_te"].reshape(-1, 1))[:, 1],
          "RF": rf_best.predict_proba(X_te)[:, 1],
          "HGB": hgb_best.predict_proba(X_te)[:, 1]}


def auc_with_interval(y, p):
    ci = bootstrap((y, p), roc_auc_score, paired=True, vectorized=False, n_resamples=500, method="percentile",
                   random_state=2026).confidence_interval
    return f"{roc_auc_score(y, p):.3f} ± {(ci.high - ci.low) / 2:.3f}"


rows = {}
for c, name in TYPES.items():
    in_c = (cluster_test == c).to_numpy()
    rows[f"{c}: {name}"] = {"customers": in_c.sum(), "default rate": f"{y_test[in_c].mean():.0%}",
                            **{model: auc_with_interval(y_test.to_numpy()[in_c], p[in_c]) for model, p in p_test.items()}}
rows["all customers"] = {"customers": len(y_test), "default rate": f"{y_test.mean():.0%}",
                         **{model: f"{roc_auc_score(y_test, p):.3f}" for model, p in p_test.items()}}
pd.DataFrame(rows).T

,customers,default rate,Logistic regression,SVM,Logit hazard,Markov debt,RF,HGB
"0: low use, pays off",1542,14%,0.699 ± 0.038,0.707 ± 0.037,0.639 ± 0.040,0.670 ± 0.036,0.715 ± 0.034,0.702 ± 0.035
"1: carries a balance, pays little",1607,9%,0.745 ± 0.043,0.748 ± 0.043,0.740 ± 0.043,0.624 ± 0.047,0.743 ± 0.039,0.752 ± 0.040
2: inactive,849,20%,0.714 ± 0.045,0.713 ± 0.045,0.645 ± 0.052,0.681 ± 0.042,0.728 ± 0.045,0.734 ± 0.043
3: often behind,1180,52%,0.711 ± 0.029,0.708 ± 0.030,0.692 ± 0.030,0.686 ± 0.029,0.712 ± 0.028,0.710 ± 0.027
4: growing balance,821,22%,0.650 ± 0.049,0.651 ± 0.050,0.626 ± 0.049,0.659 ± 0.044,0.667 ± 0.045,0.675 ± 0.047
all customers,5999,22%,0.787,0.790,0.750,0.765,0.797,0.797


Every model ranks customers less well within a type than overall, with the AUC falling from 0.75-0.80 to between
0.62 and 0.75. Within each type, the logistic regression, SVM, random forest and HGB are within about 0.02 of each
other and their intervals overlap. So the lead three models have can't be
told apart from noise within the types of customer. The two models built on monthly repayment states fall behind in
different places. The Markov chain is weakest for customers carrying a balance (0.62, against 0.75 for the
others), who are almost never late, so most of them follow the same sequence of states. The logit hazard is furthest
behind for the low-use and inactive customers (0.06 to 0.09 lower), probably because last month says the least about
these customers and it is the only month the hazard model sees. Customers with a growing balance are the hardest type
for every model, with each model scoring between 0.63 to 0.68.

## 7. Conclusion

On the test set, the random forest and HGB score highest (AUC 0.797), followed by the SVM (0.790) and the logistic
regression (0.787), with the Markov chain (0.766) and the logit hazard (0.750) further behind. The four best models
are all within 0.01 of each other overall, and within each type of customer they can't be told apart, so we can't
say that any one of them is truly best. Given this, the logistic regression does as well as the more complex models
while being the easiest to explain. The two models that only use the monthly repayment states, or only last month,
fall behind, which suggests that most of what can be predicted comes from using each customer's full six-month
history.

The within-cluster test also shows the limits of a single AUC. Every model's AUC falls from 0.75-0.80 overall to
0.62-0.75 within a type of customer, so a good part of each score comes from telling types of customer apart, rather
than ranking customers of the same type. This is limited to one test set from one six-month window, and AUC only
measures ranking, not whether the probabilities themselves are right.